In [12]:
from core.lns.operators.destroy import RandomEdgeDestroy, RandomNodeDestroy
from core.lns.operators.repair import GreedyEdgeRepair, GreedyNodeRepair, RandomRepair
from core.problems.tsp import TSPConfig, TSPSampler
from core.problems.tsp.solve import random_permutation

In [13]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Node destroy

In [14]:
n = 4
batch_size = 2

tsp_instance = TSPSampler(TSPConfig(num_locations=n), seed=42)(batch_size)
tsp_solution = random_permutation(tsp_instance)

print(tsp_solution.tour)

tensor([[3, 1, 2, 0],
        [2, 1, 3, 0]])


In [15]:
destroyed_tsp_solution = RandomNodeDestroy(2, seed=42)(tsp_solution)
print(destroyed_tsp_solution.tour)

tensor([[-1, -1,  2,  0],
        [ 2, -1, -1,  0]])


In [16]:
print(destroyed_tsp_solution.unassigned_nodes())

tensor([[1, 3],
        [1, 3]])


### Repair

In [17]:
random_repair = RandomRepair(seed=42)
_repaired = random_repair(destroyed_tsp_solution)

print(_repaired.tour)
print(_repaired.objective())

tensor([[1, 3, 2, 0],
        [2, 1, 3, 0]])
tensor([1.5279, 1.1215])


In [18]:
greedy_repair = GreedyNodeRepair()
_repaired = greedy_repair(destroyed_tsp_solution)

print(_repaired.tour)
print(_repaired.objective())

tensor([[1, 3, 2, 0],
        [2, 1, 3, 0]])
tensor([1.5279, 1.1215])


## Edge destroy

In [19]:
from core.problems.tsp.graph_solution import TSPGraphSolution

tsp_graph_solution = TSPGraphSolution.create(instance=tsp_instance, sequence=tsp_solution.tour)

print(tsp_graph_solution.prev_nodes)
print(tsp_graph_solution.next_nodes)

tensor([[2, 3, 1, 0],
        [3, 2, 0, 1]])
tensor([[3, 2, 0, 1],
        [2, 3, 1, 0]])


In [20]:
destroyed_tsp_graph_solution = RandomEdgeDestroy[TSPGraphSolution](2, seed=123)(tsp_graph_solution)
print(destroyed_tsp_graph_solution.prev_nodes)
print(destroyed_tsp_graph_solution.next_nodes)

tensor([[3, 1],
        [1, 2]])
tensor([[ 2, -1,  1, -1],
        [ 3, -1, -1,  1]])
tensor([[-1,  2,  0, -1],
        [-1,  3, -1,  0]])


## Repair

In [21]:
_repaired = GreedyEdgeRepair[TSPGraphSolution]()(destroyed_tsp_graph_solution)
print(_repaired.prev_nodes)
print(_repaired.next_nodes)

tensor([[2, 3, 1, 0],
        [3, 2, 0, 1]])
tensor([[3, 2, 0, 1],
        [2, 3, 1, 0]])
